# Camping
You are going on a camping trip, but before you leave you need to buy groceries. To optimize your time spent in the store, instead of buying the items from your shopping list in order, you plan to buy everything you need from one department before moving to the next.

Given an unsorted list of products with their departments and a shopping list, return the time saved in terms of the number of department visits eliminated.

## Approach
* convert products into a dictionary, to save on having to iterate to find a specific product
* just do a simple iteration and keep count of trips when doing in-order
* for the 2nd method, just count the number of different departments; use a set

In [ ]:
def shopping(products, shopping_list):
    product_dict = {item[0]: item[1] for item in products}
    departments = set()
    last_department = None
    trips = 0
    for item in shopping_list:
        departments.add(product_dict[item])
        if product_dict[item] != last_department:
            trips += 1
            last_department = product_dict[item]
    return trips - len(departments)

products = [
    ["Cheese",          "Dairy"],
    ["Carrots",         "Produce"],
    ["Potatoes",        "Produce"],
    ["Canned Tuna",     "Pantry"],
    ["Romaine Lettuce", "Produce"],
    ["Chocolate Milk",  "Dairy"],
    ["Flour",           "Pantry"],
    ["Iceberg Lettuce", "Produce"],
    ["Coffee",          "Pantry"],
    ["Pasta",           "Pantry"],
    ["Milk",            "Dairy"],
    ["Blueberries",     "Produce"],
    ["Pasta Sauce",     "Pantry"]
]

list1 = ["Blueberries", "Milk", "Coffee", "Flour", "Cheese", "Carrots"]
list2 = ["Blueberries", "Carrots", "Coffee", "Milk", "Flour", "Cheese"] 
list3 = ["Blueberries", "Carrots", "Romaine Lettuce", "Iceberg Lettuce"] 
list4 = ["Milk", "Flour", "Chocolate Milk", "Pasta Sauce"] 
list5 = ["Cheese", "Potatoes", "Blueberries", "Canned Tuna"]

print(shopping(products, list1)) # => 2
print(shopping(products, list2)) # => 2
print(shopping(products, list3)) # => 0
print(shopping(products, list4)) # => 2
print(shopping(products, list5)) # => 0

2
2
0
2
0


## Part 2
You and your friends are driving to a Campground to go camping. Only 2 of you have cars, so you will be carpooling.

Routes to the campground are linear, so each location will only lead to 1 location and there will be no loops or detours. Both cars will leave from their starting locations at the same time. The first car to pass someone's location will pick them up. If both cars arrive at the same time, the person can go in either car.

Roads are provided as a directed list of connected locations with the duration (in minutes) it takes to drive between the locations. [Origin, Destination, Duration it takes to drive]

Given a list of roads, a list of starting locations and a list of people/where they live, return a collection of who will be in each car upon arrival to the Campground.

### Approach
* traverse a graph
* luckily routes are linear, so I may not need to do backtracking
* I need the time to know which car reaches people to pickup first
* I'm going to convert roads and people to hash maps to make lookups a lot easier
* I'm going to simulate the `parties` traversing the roads based on the `road_map` I set up. And I'm going to use a greedy algorithm to determine which parties' turn it is with each iteration. Basically, the party with the least distance travelled will get their turn.

In [20]:
from collections import deque

def carpool(roads, starts, people):
    road_map = {road[0]: {"next":road[1], 
                          "dist": int(road[2]), 
                          "pickup": []} for road in roads}
    for person in people:
        road_map[person[1]]["pickup"].append(person[0])
    
    parties = [{"loc": starts[i], 
                "dist": deque(), 
                "dist_total": 0,
                "pickup":[]} for i in range(len(starts))]
    
    for party in parties:
        loc = party["loc"]
        while loc != "Campground":
            party["dist"].append(road_map[loc]["dist"])
            loc = road_map[loc]["next"]
    
    while parties[0]["dist"] or parties[1]["dist"]:
        if not parties[0]["dist"]:
            next_party = parties[1]
        elif not parties[1]["dist"]:
            next_party = parties[0]
        elif parties[0]["dist"][0] + parties[0]["dist_total"] < parties[1]["dist"][0] + parties[1]["dist_total"]:
            next_party = parties[0]
        else:
            next_party = parties[1]
        next_party["pickup"] += road_map[next_party["loc"]]["pickup"] 
        road_map[next_party["loc"]]["pickup"] = []
        next_party["loc"] = road_map[next_party["loc"]]["next"]
        last_dist = next_party["dist"].popleft()
        next_party["dist_total"] += last_dist
    return [party["pickup"] for party in parties] 

roads1 = [
    ["Bridgewater", "Caledonia", "30"], # <= The road from Bridgewater to Caledonia takes 30 minutes to drive.
    ["Caledonia", "New Grafton", "15"],
    ["New Grafton", "Campground", "5"],
    ["Liverpool", "Milton", "10"],
    ["Milton", "New Grafton", "30"]
]

starts1 = ["Bridgewater", "Liverpool"]

people1 = [
    ["Jessie", "Bridgewater"], ["Travis", "Caledonia"],
    ["Jeremy", "New Grafton"], ["Katie", "Liverpool"]
]

roads2 = [["Riverport", "Chester", "40"], ["Chester", "Campground", "60"], ["Halifax", "Chester", "40"]]
starts2 = ["Riverport", "Halifax"]
people2 = [["Colin", "Riverport"], ["Sam", "Chester"], ["Alyssa", "Halifax"]]

roads3 = [["Riverport", "Bridgewater", "1"], ["Bridgewater", "Liverpool", "1"], ["Liverpool", "Campground", "1"]]
starts3_1 = ["Riverport", "Bridgewater"]
starts3_2 = ["Bridgewater", "Riverport"]
starts3_3 = ["Riverport", "Liverpool"]
people3 = [["Colin", "Riverport"], ["Jessie", "Bridgewater"], ["Sam", "Liverpool"]]

print(carpool(roads1, starts1, people1)) #=> [Jessie, Travis], [Katie, Jeremy]
print(carpool(roads2, starts2, people2)) #=> [Colin, Sam], [Alyssa] OR [Colin], [Alyssa, Sam]
print(carpool(roads3, starts3_1, people3)) #=> [Colin], [Jessie, Sam]
print(carpool(roads3, starts3_2, people3)) #=> [Jessie, Sam], [Colin]
print(carpool(roads3, starts3_3, people3)) #=> [Jessie, Colin], [Sam]


[['Jessie', 'Travis'], ['Katie', 'Jeremy']]
[['Colin'], ['Alyssa', 'Sam']]
[['Colin'], ['Jessie', 'Sam']]
[['Jessie', 'Sam'], ['Colin']]
[['Colin', 'Jessie'], ['Sam']]
